<div style="text-align: center; color: red; font-size: 40px">PRETRAINED MODELS</div>

<div style="color: green; font-size: 30px">1. Wav2Vec</div>

In [ ]:
import os
import pandas as pd
import numpy as np

from transformers import Wav2Vec2Processor, Wav2Vec2Model
import torchaudio
import torch

In [ ]:
# Upload dataset from drive
from google.colab import drive
drive.mount('/content/drive')
path = "/content/drive/MyDrive/data"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Preprocess function
def preprocess(path):
  # load audio
  speech, sample_rate = torchaudio.load(path)
  # Check number of channels & convert to mono
  num_channel = speech.shape[0]
  if num_channel > 1:
    speech = torch.mean(speech, dim=0, keepdim=True)
  # Resample
  if sample_rate != 16000:
    resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
    speech = resampler(speech)
  # Flatten tensor to (sequence_length,)
  speech = speech.squeeze(0)
  return speech

In [ ]:
# Load processor and model
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/configuration_utils.py:311: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


In [ ]:
path

'/content/drive/MyDrive/data'

In [ ]:
# Get model inputs
raw_speech_list = []
sampling_rates = []

for audio in os.listdir(path):
  join = os.path.join(path, audio)
  speech, sample_rate = torchaudio.load(join)
  # Check number of channels and convert to mono before storing raw audio
  num_channel = speech.shape[0]
  if num_channel > 1:
    speech = torch.mean(speech, dim=0, keepdim=True)
  # Flatten tensor for the processor
  speech = speech.squeeze(0)
  raw_speech_list.append(speech.numpy()) # Convert to numpy as processor expects this type
  sampling_rates.append(sample_rate) # Store sampling rates

In [ ]:
inputs = processor(raw_speech_list, sampling_rate=16000, return_tensors="pt", padding=True)
# Inputs to model
with torch.no_grad():
  outputs = model(**inputs)
  hidden_states = outputs.last_hidden_state
  embedding = hidden_states.mean(dim=1)

In [ ]:
# prompt: Simple MLP

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

class EmotionClassifier(nn.Module):
  def __init__(self, input_dim, num_classes):
    super(EmotionClassifier, self).__init__()
    self.classifier = nn.Sequential(
        nn.Linear(input_dim, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, 128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, num_classes)
    )

  def forward(self, x):
    return self.classifier(x)

In [ ]:
# Instantiate classifier and extract logits
mlp = EmotionClassifier(input_dim=hidden_states.size(-1), num_classes=6)
logits = mlp(embedding)

In [ ]:
# Split dataset
train_size = int(0.8 * len(logits))
test_size = len(logits) - train_size
train_dataset, test_dataset = random_split(logits, [train_size, test_size])
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Training loop
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp.parameters(), lr=0.001)

for epoch in range(30):
  mlp.train()
  for inputs, labels in train_loader:
    with torch.no_grad():
      inputs = processor(raw_speech_list, sampling_rate=16000, return_tensors="pt", padding=True)
      outputs = model(**inputs)
      hidden_states = outputs.last_hidden_state
      embeddings = torch.mean(hidden_states, dim=1)

    logits = mlp(embeddings)
    loss = criterion(logits, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [ ]:
# Test accuracy
mlp.eval()
correct = 0
total = 0
with torch.no_grad():
  for inputs, labels in test_loader:
    inputs = processor(raw_speech_list, sampling_rate=16000, return_tensors="pt", padding=True)
    outputs = model(**inputs)
    logits = outputs.logits
    preds = torch.argmax(logits, dim=1)
    total += labels.size(0)
    correct += (preds == labels).sum().item()

  accuracy = correct / total
  print(f"Test accuracy: {accuracy:.4f}")